# BDM tamamlama defteri (eksik büyük dil modeli değerlendirmeleri)

Bu defter, **ana defterin bitmiş bir çalıştırmasını** (Drive > `NLP_Disaster/runs/<etiket>`) açar ve yalnızca **eksik kalan büyük dil modellerini** (örneğin `gpt-5-mini`, `gemini-3.8-flash`) aynı protokolle değerlendirir. Gömme modelleri, ön işleme, veri üretimi **yeniden çalıştırılmaz**; mevcut sonuçlara dokunulmaz.

Ne yapar:

1. Çalıştırma klasöründeki sentetik veri setini ve ihtiyaç listesini yükler.
2. Seçilen BDM'leri aynı sistem istemi, aynı JSON şeması ve aynı 10'arlı istek düzeniyle değerlendirir. Daha önce yanıtlanan mesajlar `data/bdm_cache` önbelleğinden alınır; yalnızca eksikler API'ye gider.
3. Hız sınırı (429) gelince bekleyip yeniden dener; **günlük sınır ya da kredi bittiyse** o modeli durdurur ve sonucu "eksik" olarak işaretler (Tablo 5'e almaz). Sorun giderilince defteri yeniden çalıştırmak yeter: kaldığı yerden devam eder.
4. **Protokol tutarlılığı:** `REDO_IDS` ile verilen mesajları (ilk denemede farklı düşünme ayarıyla yanıtlanan 210 GPT mesajı) önbellekten çıkarıp aynı ayarla yeniden ister (bir kez).
5. **Gecikme ve maliyet ölçümü (Tablo 6b):** tüm BDM'ler için aynı 150 paylaşımı 10'arlı, aynı 15 paylaşımı tek tek, sıralı çağrılarla gönderir; yalnız API süresini ölçer (hız sınırı beklemesi hariç) ve token kullanımından 1.000 paylaşım maliyetini hesaplar.
6. Tamamlanan modelleri `tablo5_bdm_*`, `tablo7_acik_ortuk_hit1`, `tablo8_anlamlilik_*` tablolarına ve `tahminler/` dosyalarına ekler (Holm düzeltmesi tüm satırlar için yeniden hesaplanır).
7. Çalıştırma klasörünü `sonuclar_<etiket>_ek.zip` olarak Drive'a yazar ve indirir.

Gerekli anahtarlar (Colab > Anahtarlar): `OPENAI_API_KEY` ve/veya `GOOGLE_API_KEY`; Claude modelleri de tamamlanacaksa `ANTHROPIC_API_KEY`.


In [ ]:
# =============================================================================
# 0. AYARLAR
# =============================================================================
DRIVE_FOLDER_NAME = "NLP_Disaster"   # Drive > MyDrive altındaki klasör
RUN_TAG = None                       # None: runs/ altındaki en yeni (smoke olmayan) çalıştırma; örn. "20261016_0743"

MODELS_TO_COMPLETE = [               # (sağlayıcı, model): yalnız bunlar değerlendirilir; anahtarı olmayan sağlayıcı atlanır
    ("openai", "gpt-5-mini"),
    ("google", "gemini-3.8-flash"),   # gemini-2.5-flash yeni kullanıcılara kapatıldı (API 404)
]
EVAL_SET_NAMES = ["sentetik"]        # değerlendirilecek veri setleri
FORCE_REEVALUATE = False             # True: bu modeller için Tablo 5'te satır olsa bile yeniden hesapla (önbellekten; API'ye yalnız eksikler gider)

# --- Protokol tutarlılığı ---------------------------------------------------------------
# İlk denemede (20261016_1116) gpt-5-mini'nin yanıtladığı 210 mesaj, düşünme ayarı (reasoning_effort) verilmeden,
# yani modelin varsayılanıyla (medium) yanıtlanmıştı; diğer 1.790 mesaj "minimal" ile. Aynı protokol için bu 210 mesaj
# önbellekten silinip "minimal" ile yeniden istenir (21 istek, ~0,01 $). Bir kez yapılır (işaret dosyası tutulur).
REDO_IDS = {("sentetik", "gpt-5-mini"): [20,21,22,23,24,25,26,27,28,29,1710,1711,1712,1713,1714,1715,1716,1717,1718,1719,1720,1721,1722,1723,1724,1725,1726,1727,1728,1729,1730,1731,1732,1733,1734,1735,1736,1737,1738,1739,1740,1741,1742,1743,1744,1745,1746,1747,1748,1749,1750,1751,1752,1753,1754,1755,1756,1757,1758,1759,1760,1761,1762,1763,1764,1765,1766,1767,1768,1769,1770,1771,1772,1773,1774,1775,1776,1777,1778,1779,1780,1781,1782,1783,1784,1785,1786,1787,1788,1789,1790,1791,1792,1793,1794,1795,1796,1797,1798,1799,1800,1801,1802,1803,1804,1805,1806,1807,1808,1809,1810,1811,1812,1813,1814,1815,1816,1817,1818,1819,1820,1821,1822,1823,1824,1825,1826,1827,1828,1829,1830,1831,1832,1833,1834,1835,1836,1837,1838,1839,1840,1841,1842,1843,1844,1845,1846,1847,1848,1849,1850,1851,1852,1853,1854,1855,1856,1857,1858,1859,1860,1861,1862,1863,1864,1865,1866,1867,1868,1869,1880,1881,1882,1883,1884,1885,1886,1887,1888,1889,1890,1891,1892,1893,1894,1895,1896,1897,1898,1899,1900,1901,1902,1903,1904,1905,1906,1907,1908,1909,1940,1941,1942,1943,1944,1945,1946,1947,1948,1949]}

# --- Gecikme ve maliyet ölçümü (Tablo 6b): tüm BDM'ler, aynı paylaşımlar, sıralı çağrılar ---------------------------
# Değerlendirme sırasındaki süreler eş zamanlı isteklerle ve (Gemini'de) hız sınırı beklemesiyle ölçüldüğü için karşılaştırılamaz;
# bu ölçüm her model için aynı 150 paylaşımı 10'arlı (toplu) ve aynı 15 paylaşımı tek tek (tekil) gönderir ve yalnız API süresini ölçer.
RUN_LATENCY_BENCHMARK = True
BENCH_MODELS = [("anthropic", "claude-opus-5-5"), ("anthropic", "claude-sonnet-5-5"), ("anthropic", "claude-haiku-5-5"),
                ("openai", "gpt-5-mini"), ("google", "gemini-3.8-flash")]
BENCH_BATCH_REQUESTS = 15
BENCH_SINGLE_REQUESTS = 15
BENCH_FORCE = False

# --- Şekil 2(b): veri setinden bir paylaşım için gerçek kosinüs puanları -------------------------------
MAKE_FIG2_EXAMPLE = True
FIG2_EXAMPLE_ID = 173                # "Families in Cebeci, Sultangazi are hungry, please send food." (gerçek ihtiyaç: gıda)
FIG2_MODEL = "sentence-transformers/all-mpnet-base-v2"   # makaledeki SBERT-mpnet

# --- Hız sınırı / kota -----------------------------------------------------------
LLM_EVAL_WORKERS = 4                                         # Anthropic eş zamanlı istek
LLM_EVAL_WORKERS_BY_PROVIDER = {"openai": 2, "google": 2}    # diğer sağlayıcılar
LLM_EVAL_RPM_BY_PROVIDER = {"openai": None, "google": 9}     # dakikadaki istek üst sınırı. None: uyarlanabilir (429 gelince bekler).
                                                             # OpenAI düşük kademedeyse (yeni hesap: 10 RPM / 50 istek-gün) "openai": 8 yazın;
                                                             # Gemini ücretsiz kademe 10 RPM, 250 istek-gün: 9 güvenli.
LLM_MIN_COVERAGE = 0.98              # mesajların en az bu oranı yanıtlanmadıysa model Tablo 5'e alınmaz ("eksik" dosyasına yazılır)
BUDGET_GUARD_USD = 10.0              # bu oturumda tahmini toplam API maliyeti bunu aşarsa yeni çağrı yapılmaz

# --- Protokol (ana defterle AYNI olmalı; önbellek imzasına girer) ------------------
LLM_EVAL_MESSAGES_PER_CALL = 10
LLM_EVAL_EFFORT = "low"              # Claude modelleri için düşünme derinliği
THINKING_MODE = "minimal"
LLM_PROMPT_VERSION = "v1"
SEED = 42
BOOTSTRAP_N = 1000
AUTO_DOWNLOAD_ZIP = True
SMOKE_TEST = False

PRICES_USD_PER_MTOK = {              # (giriş, çıkış, önbellek okuma, önbellek yazma) — maliyet tahmini için
    "claude-opus-5-5":   (4.0, 20.0, 0.20, 5.0),
    "claude-sonnet-5-5": (2.0, 10.0, 0.10, 2.5),
    "claude-haiku-5-5":  (0.10, 0.50, 0.01, 0.125),
    "gpt-5-mini":        (0.25, 2.00, 0.025, 0.0),
    "gemini-3.8-flash":  (0.75, 3.75, 0.075, 0.0),   # Google liste fiyatı (Ekim 2026; 31 Aralık 2026'ya kadar geçerli)
}

In [ ]:
# =============================================================================
# 1. KURULUM: kütüphaneler, Drive, çalıştırma klasörü, veri seti
# =============================================================================
import os, sys, re, json, time, math, random, hashlib, shutil, threading, datetime, glob, subprocess, unicodedata
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

def pip_install(pkgs):
    if os.environ.get("NLP_SKIP_PIP"):
        return
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + pkgs, check=False)

need = []
for mod, pkg in (("anthropic", "anthropic"), ("openai", "openai"), ("google.genai", "google-genai"), ("tqdm", "tqdm"), ("scipy", "scipy"), ("openpyxl", "openpyxl")):
    try:
        __import__(mod)
    except Exception:
        need.append(pkg)
if need:
    pip_install(need)

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive"
else:
    DRIVE_ROOT = os.path.abspath("./drive_local")
BASE_DIR = os.path.join(DRIVE_ROOT, "MyDrive", DRIVE_FOLDER_NAME)
DATA_DIR = os.path.join(BASE_DIR, "data")
RUNS_DIR = os.path.join(BASE_DIR, "runs")
LOCAL_SCRATCH = "/content/nlp_scratch" if IN_COLAB else os.path.abspath("./nlp_scratch")
os.makedirs(LOCAL_SCRATCH, exist_ok=True)

# --- Çalıştırma klasörü ----------------------------------------------------------
if RUN_TAG is None:
    cands = sorted(d for d in os.listdir(RUNS_DIR) if os.path.isdir(os.path.join(RUNS_DIR, d)) and not d.endswith("_smoke")
                   and os.path.exists(os.path.join(RUNS_DIR, d, "results", "sonuclar_tum_konfigurasyonlar.csv")))
    if not cands:
        raise RuntimeError(f"{RUNS_DIR} altında bitmiş bir çalıştırma bulunamadı; RUN_TAG verin.")
    RUN_TAG = cands[-1]
RUN_DIR = os.path.join(RUNS_DIR, RUN_TAG)
DIRS = {"data": DATA_DIR, "run": RUN_DIR, "results": os.path.join(RUN_DIR, "results"), "figures": os.path.join(RUN_DIR, "figures"),
        "maps": os.path.join(RUN_DIR, "maps"), "logs": os.path.join(RUN_DIR, "logs"), "preds": os.path.join(RUN_DIR, "results", "tahminler"),
        "llm_cache": os.path.join(DATA_DIR, "bdm_cache"), "scratch": LOCAL_SCRATCH}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

LOG_PATH = os.path.join(DIRS["logs"], "log.txt")
def log(*args):
    msg = " ".join(str(a) for a in args)
    stamp = datetime.datetime.now().strftime("%H:%M:%S")
    line = f"[{stamp}] {msg}"
    print(line)
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(line + "\n")
log(f"=== BDM tamamlama defteri: çalıştırma {RUN_TAG} ===")

CFG = json.load(open(os.path.join(DIRS["logs"], "config.json"), encoding="utf-8"))
FIX_NEED_TRANSLATIONS = bool(CFG.get("FIX_NEED_TRANSLATIONS", True))
if int(CFG.get("LLM_EVAL_MESSAGES_PER_CALL", LLM_EVAL_MESSAGES_PER_CALL)) != LLM_EVAL_MESSAGES_PER_CALL:
    log(f"UYARI: ana çalıştırmada LLM_EVAL_MESSAGES_PER_CALL={CFG.get('LLM_EVAL_MESSAGES_PER_CALL')}; burada {LLM_EVAL_MESSAGES_PER_CALL}. "
        "Önbellek imzası farklı olacağı için önceki yanıtlar kullanılamaz.")

# --- API anahtarları -----------------------------------------------------------
def get_secret(name):
    val = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            val = userdata.get(name)
        except Exception:
            val = None
    return val or os.environ.get(name)
ANTHROPIC_API_KEY = get_secret("ANTHROPIC_API_KEY")
OPENAI_API_KEY = get_secret("OPENAI_API_KEY")
GOOGLE_API_KEY = get_secret("GOOGLE_API_KEY")
log("Anahtarlar: Anthropic=%s | OpenAI=%s | Google=%s" % tuple("var" if k else "yok" for k in (ANTHROPIC_API_KEY, OPENAI_API_KEY, GOOGLE_API_KEY)))

# --- Yardımcılar ------------------------------------------------------------------
def tr_lower(s):
    return str(s).replace("İ", "i").replace("I", "ı").lower()
def fmt_tr(x, nd=2):
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return ""
    return f"{x:.{nd}f}".replace(".", ",")
def save_table(df, name, index=False, float_nd=3):
    p = os.path.join(DIRS["results"], name)
    df.to_csv(p + ".csv", index=index, encoding="utf-8-sig")
    try:
        df.to_excel(p + ".xlsx", index=index)
    except Exception as e:
        log("xlsx yazılamadı:", name, e)
    try:
        with open(p + ".md", "w", encoding="utf-8") as f:
            f.write(df.round(float_nd).to_markdown(index=index))
    except Exception:
        pass
TEXT_TYPE_TR = {"plain": "Düz", "lemma": "Lemma", "stem": "Gövde"}
TR_TO_TEXT_TYPE = {v: k for k, v in TEXT_TYPE_TR.items()}

# --- İhtiyaç listesi (çalıştırmadan) -----------------------------------------------
NEED_TABLE = pd.read_csv(os.path.join(DIRS["results"], "ihtiyac_listesi.csv"))
NEEDS_TR = NEED_TABLE["tr"].tolist()
TR2EN = dict(zip(NEED_TABLE["tr"], NEED_TABLE["en"]))
EN_LABELS = list(dict.fromkeys(NEED_TABLE["en"]))
EN2TR = {}
for tr, en in zip(NEED_TABLE["tr"], NEED_TABLE["en"]):
    EN2TR.setdefault(en, tr)
NEED_ALIASES_TR = {"jeneratör": "jenaratör", "jenerator": "jenaratör", "glutensiz yiyecek": "glutesiz yiyecek", "glütensiz yiyecek": "glutesiz yiyecek",
                   "power bank": "powerbank", "konteyner": "konteynır", "yara bandı": "yarabandı", "hijyenik bağ": "hijyenik ped"}
def canon_need_tr(x):
    if not isinstance(x, str):
        return None
    k = " ".join(tr_lower(x).split()); k = NEED_ALIASES_TR.get(k, k)
    return k if k in TR2EN else None

# --- Veri setleri (çalıştırmadan; yeniden üretim yok) ---------------------------------
ds_files = sorted(glob.glob(os.path.join(RUN_DIR, "data", "sentetik_paylasimlar_*.csv"))) or sorted(glob.glob(os.path.join(DATA_DIR, "sentetik_paylasimlar_*.csv")))
if not ds_files:
    raise RuntimeError("Sentetik veri seti (sentetik_paylasimlar_*.csv) bulunamadı.")
DS_PATH = ds_files[-1]
GEN_SIGNATURE = re.search(r"sentetik_paylasimlar_([0-9a-f]+)\.csv", os.path.basename(DS_PATH)).group(1)
df_syn = pd.read_csv(DS_PATH)
for c in ("label1", "label2", "label3"):
    df_syn[c] = df_syn[c].apply(lambda x: canon_need_tr(x) if isinstance(x, str) else x)
df_syn["labels_tr"] = df_syn[["label1", "label2", "label3"]].apply(lambda s: [x for x in s if isinstance(x, str)], axis=1)
df_syn["labels_en"] = df_syn["labels_tr"].apply(lambda L: list(dict.fromkeys(TR2EN[x] for x in L)))
EVAL_SETS = {"sentetik": df_syn}
log(f"Sentetik veri: {len(df_syn)} mesaj, {df_syn.district.nunique()} ilçe, imza {GEN_SIGNATURE} ({os.path.basename(DS_PATH)})")

EVAL_SETS = {k: v for k, v in EVAL_SETS.items() if k in EVAL_SET_NAMES}
log("Değerlendirilecek veri seti:", ", ".join(EVAL_SETS))

# --- Bu oturumdaki SDK sürümleri (makaledeki ortam bilgisi için) ---------------------------------------
def _ver(mod):
    try:
        import importlib.metadata as _md
        return _md.version(mod)
    except Exception:
        return None
with open(os.path.join(DIRS["logs"], "ortam_ek.txt"), "w", encoding="utf-8") as _f:
    _f.write(f"Tarih: {datetime.datetime.now().isoformat()}\n")
    for _name, _mod in (("anthropic", "anthropic"), ("openai", "openai"), ("google-genai", "google-genai"), ("sentence-transformers", "sentence-transformers")):
        _v = _ver(_mod)
        if _v:
            _f.write(f"{_name}: {_v}\n")

# --- Claude istemcisi ve maliyet sayaçları (ana defterle aynı) ------------------------

import threading
USAGE_LOCK = threading.Lock()
_API_TL = threading.local()   # son başarılı API çağrısının süresi (ms): bekleme/yeniden deneme süreleri hariç
USAGE = defaultdict(lambda: Counter())
class BudgetExceeded(RuntimeError):
    pass
_BUDGET_LOGGED = [False]
def add_usage(model, usage):
    if usage is None:
        return
    with USAGE_LOCK:
        for k in ("input_tokens", "output_tokens", "cache_creation_input_tokens", "cache_read_input_tokens"):
            USAGE[model][k] += int(getattr(usage, k, 0) or 0)
        det = getattr(usage, "output_tokens_details", None)
        USAGE[model]["thinking_tokens"] += int(getattr(det, "thinking_tokens", 0) or 0) if det is not None else 0
        USAGE[model]["calls"] += 1
def usage_cost_usd(model):
    u = USAGE.get(model, {})
    p = PRICES_USD_PER_MTOK.get(model)
    if not p:
        return float("nan")
    return (u.get("input_tokens", 0) * p[0] + u.get("output_tokens", 0) * p[1] +
            u.get("cache_read_input_tokens", 0) * p[2] + u.get("cache_creation_input_tokens", 0) * p[3]) / 1e6
def total_cost_usd():
    return float(np.nansum([usage_cost_usd(m) for m in USAGE])) if USAGE else 0.0
def usage_summary(model):
    u = USAGE.get(model, {}); n = max(1, u.get("calls", 0))
    return (f"{u.get('calls', 0)} çağrı | çağrı başına ort. çıktı {u.get('output_tokens', 0) / n:.0f} token "
            f"(düşünme {u.get('thinking_tokens', 0) / n:.0f}) | giriş {u.get('input_tokens', 0) / n:.0f} + önbellek {u.get('cache_read_input_tokens', 0) / n:.0f} | "
            f"tahmini ${usage_cost_usd(model):.2f}")
def check_budget():
    if BUDGET_GUARD_USD is not None and total_cost_usd() >= BUDGET_GUARD_USD:
        if not _BUDGET_LOGGED[0]:
            _BUDGET_LOGGED[0] = True
            log(f"BÜTÇE FRENİ: tahmini maliyet ${total_cost_usd():.2f} >= {BUDGET_GUARD_USD} $. Yeni API çağrısı yapılmıyor; "
                f"devam etmek için BUDGET_GUARD_USD'yi artırıp hücreyi yeniden çalıştırın (tamamlanan parçalar Drive'da).")
        raise BudgetExceeded("bütçe aşıldı")

def thinking_param(model, mode):
    """Düşünme (thinking) denetimi: Sonnet 5.5 ön düşünmeyi 'between_tools' ile kapatır, Haiku 5.5 'disabled' kabul eder,
    Opus 5.5 / Fable 5.1'de düşünme kapatılamaz (yalnız effort ile sınırlanır)."""
    if mode != "minimal":
        return None
    if "sonnet-5-5" in model:
        return {"type": "between_tools"}
    if "haiku-5-5" in model:
        return {"type": "disabled"}
    return None

def extract_json(text):
    try:
        return json.loads(text)
    except Exception:
        s, e = text.find("{"), text.rfind("}")
        return json.loads(text[s:e + 1])

ANTHROPIC_CLIENT = None
SDK_SUPPORTS = {"temperature": False, "output_config": True}
if not SMOKE_TEST and ANTHROPIC_API_KEY:
    import anthropic, inspect as _inspect
    ANTHROPIC_CLIENT = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY, max_retries=4, timeout=600)
    try:
        _params = set(_inspect.signature(ANTHROPIC_CLIENT.messages.create).parameters)
        SDK_SUPPORTS = {"temperature": "temperature" in _params, "output_config": "output_config" in _params}
    except Exception:
        pass
    log(f"anthropic SDK {anthropic.__version__}: temperature={'var' if SDK_SUPPORTS['temperature'] else 'yok (API varsayılanı kullanılır)'}, "
        f"output_config={'var' if SDK_SUPPORTS['output_config'] else 'yok (düz JSON istemi)'}")

def claude_json(system_text, user_text, schema, model, max_tokens=8000, temperature=None, retries=6, effort=None, thinking_mode=None):
    """Yapılandırılmış JSON çıktısı (output_config.format). Düşünme/effort denetimi, SDK imzasına göre parametre seçimi,
    bütçe freni, kesilen yanıtta max_tokens artırımı, 429/5xx için üstel bekleme. Desteklenmeyen parametre 400 verirse düşürülür."""
    last_err = None
    thinking_mode = THINKING_MODE if thinking_mode is None else thinking_mode
    use_temp = SDK_SUPPORTS["temperature"] and temperature is not None
    use_output_config = SDK_SUPPORTS["output_config"]
    think = thinking_param(model, thinking_mode)
    eff = effort
    for attempt in range(retries):
        check_budget()
        try:
            kwargs = dict(model=model, max_tokens=max_tokens,
                          system=[{"type": "text", "text": system_text, "cache_control": {"type": "ephemeral"}}],
                          messages=[{"role": "user", "content": user_text}])
            if use_temp:
                kwargs["temperature"] = temperature
            oc = {}
            if use_output_config:
                oc["format"] = {"type": "json_schema", "schema": schema}
            else:
                kwargs["messages"][0]["content"] += "\n\nReturn ONLY valid JSON matching this schema:\n" + json.dumps(schema)
            if eff and use_output_config:
                oc["effort"] = eff
            if oc:
                kwargs["output_config"] = oc
            if think:
                kwargs["thinking"] = think
            _t0 = time.perf_counter()
            resp = ANTHROPIC_CLIENT.messages.create(**kwargs)
            _API_TL.last_ms = (time.perf_counter() - _t0) * 1000
            add_usage(model, getattr(resp, "usage", None))
            if resp.stop_reason == "max_tokens":
                max_tokens = int(max_tokens * 1.5)
                raise RuntimeError(f"Yanıt max_tokens sınırında kesildi, sınır {max_tokens} olarak artırıldı")
            if resp.stop_reason == "refusal":
                raise RuntimeError("Model yanıtı reddetti (refusal)")
            text = "".join(b.text for b in resp.content if getattr(b, "type", "") == "text")
            return extract_json(text)
        except BudgetExceeded:
            raise
        except TypeError as e:   # SDK imzası beklenenden farklı: parametreyi düşür ve hemen yeniden dene
            last_err = e; msg = str(e)
            if "temperature" in msg and use_temp:
                log("SDK temperature parametresini kabul etmiyor, parametresiz devam ediliyor"); use_temp = False; SDK_SUPPORTS["temperature"] = False; continue
            if "output_config" in msg and use_output_config:
                log("SDK output_config parametresini kabul etmiyor, düz JSON istemine geçiliyor"); use_output_config = False; SDK_SUPPORTS["output_config"] = False; continue
            if "thinking" in msg and think:
                log("SDK thinking parametresini kabul etmiyor, parametresiz devam ediliyor"); think = None; continue
            raise
        except Exception as e:
            last_err = e; msg = str(e)
            if e.__class__.__name__ == "BadRequestError":
                if use_temp and "temperature" in msg:
                    log(f"{model}: API temperature kabul etmedi, parametresiz yeniden deneniyor"); use_temp = False; continue
                if think and "thinking" in msg:
                    log(f"{model}: thinking={think} kabul edilmedi ({msg[:120]}), parametresiz yeniden deneniyor"); think = None; continue
                if eff and "effort" in msg:
                    log(f"{model}: effort={eff} kabul edilmedi ({msg[:120]}), effort olmadan yeniden deneniyor"); eff = None; continue
                raise  # diğer 400 hatalarında yeniden denemenin anlamı yok
            wait = min(90, (2 ** attempt) + random.random() * 2)
            log(f"API hatası ({model}, deneme {attempt + 1}/{retries}): {msg[:160]} -> {wait:.0f} sn bekleniyor")
            time.sleep(wait)
    raise RuntimeError(f"API çağrısı başarısız: {last_err}")


# --- Ölçütler (ana defterden) ---
# --- Ölçütler -----------------------------------------------------------------------------
def top_modes(counter):
    if not counter:
        return set()
    m = max(counter.values())
    return {k for k, v in counter.items() if v == m}

def district_metric_paper(df, preds):
    """Makaledeki ölçüt: ilçede en sık tahmin edilen ihtiyaç(lar) ile en sık gerçek ihtiyaç(lar) kesişiyorsa 1, değilse 0."""
    acc = {}
    for d, idx in df.groupby("district").indices.items():
        actual = Counter(x for i in idx for x in df["labels_en"].iat[i])
        predicted = Counter(preds[i] for i in idx if preds[i] is not None)
        if not actual or not predicted:
            continue
        acc[d] = 1 if (top_modes(actual) & top_modes(predicted)) else 0
    return acc

def district_top_pred(df, preds):
    out = {}
    for d, idx in df.groupby("district").indices.items():
        c = Counter(preds[i] for i in idx if preds[i] is not None)
        if c:
            out[d] = c.most_common(1)[0]
    return out

def bootstrap_ci(values, n_boot=BOOTSTRAP_N, seed=SEED):
    v = np.asarray(values, dtype=float)
    if len(v) == 0:
        return (float("nan"), float("nan"))
    rng = np.random.default_rng(seed)
    means = rng.choice(v, size=(n_boot, len(v)), replace=True).mean(axis=1)
    return (float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5)))

def evaluate_predictions(df, top1, top3, tag):
    labels = df["labels_en"].tolist()
    hit1 = np.array([(p is not None) and (p in L) for p, L in zip(top1, labels)], dtype=float)
    hit3 = np.array([any((p in L) for p in P if p is not None) for P, L in zip(top3, labels)], dtype=float)
    d_acc = district_metric_paper(df, top1)
    d_vals = list(d_acc.values())
    hit1_by_d = pd.Series(hit1, index=df["district"].values).groupby(level=0).mean()
    ci_d, ci_m = bootstrap_ci(d_vals), bootstrap_ci(hit1)
    res = {"ilce_dogruluk_makale": float(np.mean(d_vals)) if d_vals else float("nan"), "ilce_ga_alt": ci_d[0], "ilce_ga_ust": ci_d[1],
           "hit1": float(hit1.mean()), "hit1_ga_alt": ci_m[0], "hit1_ga_ust": ci_m[1],
           "hit1_ilce_ort": float(hit1_by_d.mean()), "hit3": float(hit3.mean()),
           "n_ilce": len(d_vals), "n_mesaj": len(df), "tahminsiz": int(sum(p is None for p in top1))}
    dtp = district_top_pred(df, top1)
    rows = []
    for d, idx in df.groupby("district").indices.items():
        actual = Counter(x for i in idx for x in df["labels_en"].iat[i])
        rows.append({"İlçe": d, "mesaj": len(idx), "gercek_en_sik": ", ".join(sorted(top_modes(actual))),
                     "tahmin_en_sik": dtp.get(d, ("", 0))[0], "tahmin_sayi": dtp.get(d, ("", 0))[1],
                     "dogru_makale_olcutu": d_acc.get(d, np.nan), "hit1_ort": float(hit1_by_d.get(d, np.nan))})
    pd.DataFrame(rows).to_csv(os.path.join(DIRS["preds"], f"ilce_{tag}.csv"), index=False, encoding="utf-8-sig")
    pd.DataFrame({"id": df["id"].values, "district": df["district"].values, "gercek": [";".join(L) for L in labels],
                  "top1": top1, "top2": [P[1] if len(P) > 1 else None for P in top3], "top3": [P[2] if len(P) > 2 else None for P in top3],
                  "hit1": hit1.astype(int)}).to_csv(os.path.join(DIRS["preds"], f"mesaj_{tag}.csv"), index=False, encoding="utf-8-sig")
    return res, hit1


# --- Eşleştirilmiş testler (ana defterden) ---
# --- Eşleştirilmiş anlamlılık testleri (aynı mesajlar, ikili sonuçlar) -------------------------------
def mcnemar_exact(a, b):
    """a, b: 0/1 isabet vektörleri. Uyuşmayan çiftler üzerinde tam binom testi (iki yönlü)."""
    n10 = int(((a == 1) & (b == 0)).sum()); n01 = int(((a == 0) & (b == 1)).sum())
    n = n10 + n01
    if n == 0:
        return 1.0, n10, n01
    try:
        from scipy.stats import binomtest
        return float(binomtest(min(n10, n01), n, 0.5, alternative="two-sided").pvalue), n10, n01
    except Exception:
        from math import comb
        k = min(n10, n01); p = sum(comb(n, i) for i in range(k + 1)) / 2 ** n
        return float(min(1.0, 2 * p)), n10, n01

def paired_bootstrap(a, b, n_boot=2000, seed=SEED):
    rng = np.random.default_rng(seed); n = len(a); d = a - b
    idx = rng.integers(0, n, size=(n_boot, n))
    diffs = d[idx].mean(axis=1)
    return float(d.mean()), float(np.percentile(diffs, 2.5)), float(np.percentile(diffs, 97.5))

def holm(pvals):
    p = np.asarray(pvals, dtype=float); m = len(p); order = np.argsort(p); adj = np.empty(m)
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (m - rank) * p[i]); adj[i] = min(1.0, running)
    return adj

In [ ]:
# =============================================================================
# 2. BDM DEĞERLENDİRME KODU (ana defterle birebir aynı istem, şema ve hız/kota yönetimi)
# =============================================================================
import difflib
LLM_PROMPT_VERSION = "v1"
SYSTEM_PROMPT_LLM = ("You are a disaster response assistant. Based on the social media message of an earthquake victim, "
                     "identify the single most urgently needed relief item from the following list of disaster relief needs:\n"
                     + "\n".join(f"- {n}" for n in EN_LABELS) +
                     "\nFor each message respond only with the name of the need, exactly as written in the list.")
LLM_SCHEMA = {"type": "object", "properties": {"predictions": {"type": "array", "items": {
    "type": "object", "properties": {"id": {"type": "integer"}, "need": {"type": "string", "enum": EN_LABELS}},
    "required": ["id", "need"], "additionalProperties": False}}}, "required": ["predictions"], "additionalProperties": False}
with open(os.path.join(DIRS["logs"], "prompt_bdm.txt"), "w", encoding="utf-8") as f:
    f.write(SYSTEM_PROMPT_LLM)

def strip_additional(schema):
    if isinstance(schema, dict):
        return {k: strip_additional(v) for k, v in schema.items() if k != "additionalProperties"}
    if isinstance(schema, list):
        return [strip_additional(x) for x in schema]
    return schema

_LABEL_LOWER = {l.lower(): l for l in EN_LABELS}
def normalize_need(x):
    if not isinstance(x, str):
        return None
    k = x.strip().lower()
    if k in _LABEL_LOWER:
        return _LABEL_LOWER[k]
    m = difflib.get_close_matches(k, list(_LABEL_LOWER), n=1, cutoff=0.6)
    return _LABEL_LOWER[m[0]] if m else None

def build_llm_user(batch):
    if len(batch) == 1:
        return f"Message (id {batch[0][0]}): {batch[0][1]}\nReturn JSON {{\"predictions\":[{{\"id\":{batch[0][0]},\"need\":...}}]}}."
    return "Messages:\n" + "\n".join(f"[id {i}] {t}" for i, t in batch) + "\nReturn JSON {\"predictions\":[{\"id\":..,\"need\":..}, ...]} with one entry per message id."

# --- Hız sınırı (429) ve kota yönetimi -------------------------------------------------------
class LLMQuotaExhausted(Exception):
    """Günlük istek sınırı ya da kredi/kota bitti: bu model için yeni istek göndermenin anlamı yok."""

class _Pacer:
    """Sağlayıcı başına istek aralığı (RPM) ve 429 sonrası ortak bekleme."""
    def __init__(self, rpm=None):
        self.lock = threading.Lock(); self.next_ok = 0.0; self.min_interval = (60.0 / rpm) if rpm else 0.0
    def wait(self):
        with self.lock:
            now = time.monotonic(); delay = max(0.0, self.next_ok - now)
            self.next_ok = max(now, self.next_ok) + self.min_interval
        if delay > 0:
            time.sleep(delay)
    def hold(self, seconds):
        with self.lock:
            self.next_ok = max(self.next_ok, time.monotonic() + seconds)

_PACERS = {}
def pacer_for(provider):
    if provider not in _PACERS:
        _PACERS[provider] = _Pacer(LLM_EVAL_RPM_BY_PROVIDER.get(provider))
    return _PACERS[provider]

def _retry_after_seconds(msg):
    m = re.search(r"try again in\s*([\d.]+)\s*(ms|s|m)\b", msg, flags=re.I)
    if not m:
        return None
    v = float(m.group(1)); u = m.group(2).lower()
    return v / 1000 if u == "ms" else (v * 60 if u == "m" else v)

def classify_api_error(e):
    """'quota' (kredi/kota bitti), 'daily' (günlük istek sınırı), 'rate' (dakikalık sınır), 'other'."""
    msg = str(e); low = msg.lower()
    is429 = ("429" in msg) or ("rate limit" in low) or ("resource_exhausted" in low) or ("quota" in low) or type(e).__name__ in ("RateLimitError", "ResourceExhausted")
    if any(k in low for k in ("insufficient_quota", "no credits remaining", "credit_balance_exhausted", "spend_limit_exceeded", "usage_limit_exceeded", "billing")):
        return "quota"
    if is429 and any(k in low for k in ("per day", "requests per day", "tokens per day", "rpd", "perday", "daily")):
        return "daily"
    if is429:
        return "rate"
    return "other"

def llm_call_paced(provider, model, batch, max_attempts=8):
    """llm_call'ı hız sınırına uyarak çağırır: dakikalık 429'da bekleyip yeniden dener; kota/günlük sınırda LLMQuotaExhausted fırlatır."""
    pacer = pacer_for(provider)
    for attempt in range(max_attempts):
        if not SMOKE_TEST:
            pacer.wait()
        try:
            _API_TL.last_ms = float("nan")
            preds = llm_call(provider, model, batch)
            return preds, float(getattr(_API_TL, "last_ms", float("nan")))
        except LLMQuotaExhausted:
            raise
        except BudgetExceeded:
            raise
        except Exception as e:
            kind = classify_api_error(e)
            if kind in ("quota", "daily"):
                raise LLMQuotaExhausted(f"{kind}: {str(e)[:300]}")
            wait_s = _retry_after_seconds(str(e))
            if kind == "rate":
                wait_s = wait_s if wait_s is not None else min(60.0, 3.0 * (2 ** attempt))
                pacer.hold(wait_s + 0.5)
                time.sleep(wait_s + 0.5 + random.random())
                continue
            if attempt >= 2:
                raise
            time.sleep(2.0 * (attempt + 1))
    raise RuntimeError(f"{model}: {max_attempts} denemede hız sınırı aşılamadı")

_PROVIDER_OPTS = {"openai_reasoning": "minimal", "google_thinking_budget": 0}
OTHER_USAGE = defaultdict(lambda: Counter())
def _note_usage(model, u):
    """OpenAI / Google token sayaçları. Faturalanan çıktı: OpenAI'de completion_tokens (düşünme dâhil), Gemini'de candidates + thoughts."""
    if u is None:
        return
    with USAGE_LOCK:
        c = OTHER_USAGE[model]; c["calls"] += 1
        if hasattr(u, "prompt_tokens"):            # OpenAI
            det = getattr(u, "completion_tokens_details", None); pdt = getattr(u, "prompt_tokens_details", None)
            out = int(getattr(u, "completion_tokens", 0) or 0); think = int(getattr(det, "reasoning_tokens", 0) or 0) if det is not None else 0
            c["input_tokens"] += int(getattr(u, "prompt_tokens", 0) or 0); c["cached_input_tokens"] += int(getattr(pdt, "cached_tokens", 0) or 0) if pdt is not None else 0
            c["output_tokens"] += out - think; c["thinking_tokens"] += think; c["billed_output_tokens"] += out
        else:                                       # Google
            out = int(getattr(u, "candidates_token_count", 0) or 0); think = int(getattr(u, "thoughts_token_count", 0) or 0)
            c["input_tokens"] += int(getattr(u, "prompt_token_count", 0) or 0); c["cached_input_tokens"] += int(getattr(u, "cached_content_token_count", 0) or 0)
            c["output_tokens"] += out; c["thinking_tokens"] += think; c["billed_output_tokens"] += out + think

def cost_from_counter(model, u):
    """Sayaç sözlüğünden tahmini maliyet (USD, liste fiyatı)."""
    p = PRICES_USD_PER_MTOK.get(model)
    if not p or not u:
        return float("nan")
    if model.startswith("claude"):
        return (u.get("input_tokens", 0) * p[0] + u.get("output_tokens", 0) * p[1] + u.get("cache_read_input_tokens", 0) * p[2] + u.get("cache_creation_input_tokens", 0) * p[3]) / 1e6
    cached = u.get("cached_input_tokens", 0)
    return ((u.get("input_tokens", 0) - cached) * p[0] + cached * p[2] + u.get("billed_output_tokens", u.get("output_tokens", 0)) * p[1]) / 1e6

def other_usage_summary(model):
    u = OTHER_USAGE.get(model, {}); n = max(1, u.get("calls", 0)); cost = cost_from_counter(model, u)
    return (f"{u.get('calls', 0)} çağrı | çağrı başına giriş {u.get('input_tokens', 0) / n:.0f} (önbellek {u.get('cached_input_tokens', 0) / n:.0f}), çıktı {u.get('output_tokens', 0) / n:.0f} token "
            f"(düşünme {u.get('thinking_tokens', 0) / n:.0f})" + (f" | tahmini ${cost:.2f}" if cost == cost else ""))
def other_usage_cost(model):
    return cost_from_counter(model, OTHER_USAGE.get(model, {}))

_OPENAI_CLIENT = _GOOGLE_CLIENT = None
def llm_call(provider, model, batch):
    """batch: [(id, text), ...] -> {id: need}"""
    global _OPENAI_CLIENT, _GOOGLE_CLIENT
    user = build_llm_user(batch)
    if SMOKE_TEST:
        _API_TL.last_ms = 0.0
        rng = random.Random(batch[0][0] + hash(model) % 1000)
        return {i: rng.choice(EN_LABELS) for i, _ in batch}
    if provider == "anthropic":
        data = claude_json(SYSTEM_PROMPT_LLM, user, LLM_SCHEMA, model, max_tokens=150 * len(batch) + 1500, temperature=0.0, effort=LLM_EVAL_EFFORT)
    elif provider == "openai":
        if _OPENAI_CLIENT is None:
            from openai import OpenAI
            _OPENAI_CLIENT = OpenAI(api_key=OPENAI_API_KEY, max_retries=0)
        kwargs = dict(model=model, messages=[{"role": "system", "content": SYSTEM_PROMPT_LLM}, {"role": "user", "content": user}],
                      response_format={"type": "json_schema", "json_schema": {"name": "preds", "strict": True, "schema": LLM_SCHEMA}})
        if _PROVIDER_OPTS["openai_reasoning"] and (model.startswith("gpt-5") or model.startswith("o")):
            kwargs["reasoning_effort"] = _PROVIDER_OPTS["openai_reasoning"]   # en düşük düşünme: ucuz ve hızlı (Claude'daki effort="low" karşılığı)
        try:
            _t0 = time.perf_counter(); r = _OPENAI_CLIENT.chat.completions.create(**kwargs); _API_TL.last_ms = (time.perf_counter() - _t0) * 1000
        except Exception as e:
            if "reasoning_effort" in kwargs and ("reasoning" in str(e).lower() or "unsupported" in str(e).lower()) and "429" not in str(e):
                log(f"{model}: reasoning_effort kabul edilmedi ({str(e)[:100]}); parametresiz devam"); _PROVIDER_OPTS["openai_reasoning"] = None
                kwargs.pop("reasoning_effort"); _t0 = time.perf_counter(); r = _OPENAI_CLIENT.chat.completions.create(**kwargs); _API_TL.last_ms = (time.perf_counter() - _t0) * 1000
            else:
                raise
        _note_usage(model, getattr(r, "usage", None))
        data = extract_json(r.choices[0].message.content)
    elif provider == "google":
        if _GOOGLE_CLIENT is None:
            from google import genai
            _GOOGLE_CLIENT = genai.Client(api_key=GOOGLE_API_KEY)
        cfg = {"system_instruction": SYSTEM_PROMPT_LLM, "response_mime_type": "application/json", "response_schema": strip_additional(LLM_SCHEMA), "temperature": 0.0}
        if _PROVIDER_OPTS["google_thinking_budget"] is not None:
            cfg["thinking_config"] = {"thinking_budget": _PROVIDER_OPTS["google_thinking_budget"]}   # 0: düşünme kapalı (2.5 Flash)
        try:
            _t0 = time.perf_counter(); r = _GOOGLE_CLIENT.models.generate_content(model=model, contents=user, config=cfg); _API_TL.last_ms = (time.perf_counter() - _t0) * 1000
        except Exception as e:
            if "thinking_config" in cfg and "thinking" in str(e).lower() and "429" not in str(e):
                log(f"{model}: thinking_budget kabul edilmedi ({str(e)[:100]}); parametresiz devam"); _PROVIDER_OPTS["google_thinking_budget"] = None
                cfg.pop("thinking_config"); _t0 = time.perf_counter(); r = _GOOGLE_CLIENT.models.generate_content(model=model, contents=user, config=cfg); _API_TL.last_ms = (time.perf_counter() - _t0) * 1000
            else:
                raise
        _note_usage(model, getattr(r, "usage_metadata", None))
        data = extract_json(r.text)
    else:
        raise ValueError(provider)
    return {int(p["id"]): normalize_need(p.get("need")) for p in data.get("predictions", [])}

def llm_cache_path(provider, model, set_name):
    sig = hashlib.md5(json.dumps([set_name, GEN_SIGNATURE, LLM_EVAL_MESSAGES_PER_CALL, LLM_PROMPT_VERSION, FIX_NEED_TRANSLATIONS]).encode()).hexdigest()[:8]
    return os.path.join(DIRS["llm_cache"], f"{set_name}_{provider}_{model}_{sig}.jsonl")

def llm_proto_tag(provider, model):
    """Önbellek satırına yazılan protokol etiketi (denetim için)."""
    if provider == "anthropic":
        return f"effort={LLM_EVAL_EFFORT};thinking={thinking_param(model, THINKING_MODE)}"
    if provider == "openai":
        return f"reasoning_effort={_PROVIDER_OPTS['openai_reasoning']}"
    if provider == "google":
        return f"thinking_budget={_PROVIDER_OPTS['google_thinking_budget']};temperature=0"
    return ""

def llm_evaluate(provider, model, dfe, set_name):
    """Dönen: top1 listesi, medyan ms/mesaj, başarısız çağrı sayısı, tahminsiz mesaj sayısı, durma nedeni (None: tamamlandı)."""
    ids, texts = dfe["id"].tolist(), dfe["English"].tolist()
    cache_path = llm_cache_path(provider, model, set_name)
    cache = {}
    if os.path.exists(cache_path):
        with open(cache_path, encoding="utf-8") as f:
            for line in f:
                try:
                    r = json.loads(line)
                    if r.get("need") is not None:
                        cache[int(r["id"])] = r
                except Exception:
                    pass
    todo = [(i, t) for i, t in zip(ids, texts) if i not in cache]
    batches = [todo[k:k + LLM_EVAL_MESSAGES_PER_CALL] for k in range(0, len(todo), LLM_EVAL_MESSAGES_PER_CALL)]
    if cache:
        log(f"{model} ({set_name}): {len(cache)} mesaj önbellekten, {len(todo)} mesaj ({len(batches)} istek) API'den")
    lock = threading.Lock(); n_fail = 0; n_ok = 0; stop_reason = None
    workers = LLM_EVAL_WORKERS_BY_PROVIDER.get(provider, LLM_EVAL_WORKERS) if provider != "anthropic" else LLM_EVAL_WORKERS
    proto = llm_proto_tag(provider, model)
    def work(batch):
        preds, api_ms = llm_call_paced(provider, model, batch); ms = api_ms / len(batch)   # yalnız API süresi (hız sınırı beklemesi hariç)
        return [{"id": i, "need": preds.get(i), "ms": ms, "proto": proto} for i, _ in batch]
    if batches:
        ex = ThreadPoolExecutor(max_workers=workers)
        fout = open(cache_path, "a", encoding="utf-8")
        futs = [ex.submit(work, b) for b in batches]
        bar = tqdm(total=len(futs), desc=f"BDM {model} ({set_name})")
        try:
            for fut in as_completed(futs):
                try:
                    rows = fut.result()
                except (LLMQuotaExhausted, BudgetExceeded) as e:
                    if stop_reason is None:
                        stop_reason = str(e)[:300]
                        log(f"DURDURULDU ({model}, {set_name}): {stop_reason}")
                        ex.shutdown(wait=False, cancel_futures=True)
                    n_fail += 1
                except Exception as e:
                    n_fail += 1; log(f"BDM çağrısı başarısız ({model}): {str(e)[:200]}")
                else:
                    n_ok += 1
                    with lock:
                        for r in rows:
                            if r["need"] is not None:
                                cache[r["id"]] = r
                            fout.write(json.dumps(r, ensure_ascii=False) + "\n")
                bar.update(1); bar.set_postfix({"başarılı": n_ok, "başarısız": n_fail})
                if stop_reason is not None and all(f.done() or f.cancelled() for f in futs):
                    break
        finally:
            bar.close(); fout.close(); ex.shutdown(wait=False, cancel_futures=True)
    top1 = [cache.get(i, {}).get("need") for i in ids]
    ms = [cache[i]["ms"] for i in ids if i in cache and cache[i].get("ms") is not None]
    n_missing = len(ids) - sum(p is not None for p in top1)
    return top1, (float(np.median(ms)) if ms else float("nan")), n_fail, n_missing, stop_reason

# --- Gecikme ve maliyet ölçümü: tüm BDM'ler için AYNI paylaşımlar, sıralı (tek iş parçacığı) çağrılar ---------------
def _usage_snapshot(provider, model):
    src = USAGE if provider == "anthropic" else OTHER_USAGE
    return dict(src.get(model, {}))

def _usage_delta(before, after):
    return {k: after.get(k, 0) - before.get(k, 0) for k in set(before) | set(after)}

def llm_latency_benchmark(provider, model, dfe, n_batch_req, n_single_req, force=False):
    """Toplu (LLM_EVAL_MESSAGES_PER_CALL paylaşım/istek) ve tekil (1 paylaşım/istek) kipte gecikme ve token kullanımı.
    Paylaşımlar sabit tohumla seçilir (tüm modellerde aynı). Hız sınırı beklemesi ölçüme girmez. Sonuç Drive'da önbelleğe alınır."""
    mpc = LLM_EVAL_MESSAGES_PER_CALL
    bench_dir = os.path.join(DATA_DIR, "bdm_bench"); os.makedirs(bench_dir, exist_ok=True)
    sig = hashlib.md5(json.dumps([GEN_SIGNATURE, mpc, n_batch_req, n_single_req, LLM_PROMPT_VERSION, llm_proto_tag(provider, model)]).encode()).hexdigest()[:8]
    out_path = os.path.join(bench_dir, f"{provider}_{model}_{sig}.json")
    if os.path.exists(out_path) and not force:
        return json.load(open(out_path, encoding="utf-8"))
    rng = np.random.default_rng(SEED + 2026)
    pool = rng.choice(dfe["id"].values, size=min(len(dfe), n_batch_req * mpc + n_single_req), replace=False)
    text = dict(zip(dfe["id"].tolist(), dfe["English"].tolist()))
    batch_reqs = [[(int(i), text[int(i)]) for i in pool[k * mpc:(k + 1) * mpc]] for k in range(n_batch_req)]
    single_reqs = [[(int(i), text[int(i)])] for i in pool[n_batch_req * mpc:n_batch_req * mpc + n_single_req]]
    res = {"model": model, "saglayici": provider, "protokol": llm_proto_tag(provider, model), "tamam": True, "neden": None}
    for mode, reqs in (("toplu", batch_reqs), ("tekil", single_reqs)):
        times, n_posts, n_ok = [], 0, 0
        before = _usage_snapshot(provider, model)
        for req in tqdm(reqs, desc=f"Ölçüm {model} ({mode})", leave=False):
            try:
                preds, api_ms = llm_call_paced(provider, model, req)
            except (LLMQuotaExhausted, BudgetExceeded) as e:
                res["tamam"] = False; res["neden"] = str(e)[:200]; log(f"Ölçüm durdu ({model}, {mode}): {res['neden']}"); break
            except Exception as e:
                log(f"Ölçüm çağrısı başarısız ({model}, {mode}): {str(e)[:150]}"); continue
            times.append(api_ms); n_posts += len(req); n_ok += sum(preds.get(i) is not None for i, _ in req)
        u = _usage_delta(before, _usage_snapshot(provider, model))
        cost = cost_from_counter(model, u)
        t = np.array([x for x in times if x == x], dtype=float); k = max(1, len(times))
        res[mode] = {"istek": len(times), "paylasim": n_posts, "yanitlanan": n_ok,
                     "ms_istek_medyan": float(np.median(t)) if len(t) else float("nan"), "ms_istek_p90": float(np.percentile(t, 90)) if len(t) else float("nan"),
                     "ms_paylasim_medyan": float(np.median(t)) / (mpc if mode == "toplu" else 1) if len(t) else float("nan"),
                     "giris_token_istek": u.get("input_tokens", 0) / k, "cikis_token_istek": (u.get("billed_output_tokens", u.get("output_tokens", 0))) / k,
                     "dusunme_token_istek": u.get("thinking_tokens", 0) / k,
                     "usd": cost, "usd_1000_paylasim": cost / n_posts * 1000 if n_posts else float("nan")}
        if not res["tamam"]:
            break
    if res["tamam"]:
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(res, f, ensure_ascii=False, indent=1)
    return res

def run_latency_benchmarks(models, dfe):
    rows = []
    for provider, model in models:
        key = {"anthropic": ANTHROPIC_API_KEY, "openai": OPENAI_API_KEY, "google": GOOGLE_API_KEY}.get(provider)
        if not key and not SMOKE_TEST:
            log(f"Ölçüm: {model} atlandı (anahtar yok)"); continue
        r = llm_latency_benchmark(provider, model, dfe, BENCH_BATCH_REQUESTS, BENCH_SINGLE_REQUESTS, force=BENCH_FORCE)
        row = {"Model": model, "Sağlayıcı": provider, "Protokol": r.get("protokol"), "Tamam": r.get("tamam")}
        for mode in ("toplu", "tekil"):
            m = r.get(mode, {})
            row.update({f"{mode}_istek": m.get("istek"), f"{mode}_ms_istek_medyan": m.get("ms_istek_medyan"), f"{mode}_ms_istek_p90": m.get("ms_istek_p90"),
                        f"{mode}_ms_paylasim": m.get("ms_paylasim_medyan"), f"{mode}_giris_token": m.get("giris_token_istek"),
                        f"{mode}_cikis_token": m.get("cikis_token_istek"), f"{mode}_dusunme_token": m.get("dusunme_token_istek"),
                        f"{mode}_usd_1000_paylasim": m.get("usd_1000_paylasim")})
        rows.append(row)
        def _f(x, nd=0):
            return "?" if x is None or x != x else f"{x:.{nd}f}"
        log(f"Ölçüm {model}: toplu {_f(row.get('toplu_ms_paylasim'))} ms/paylaşım (istek medyanı {_f(row.get('toplu_ms_istek_medyan'))} ms), "
            f"tekil {_f(row.get('tekil_ms_istek_medyan'))} ms, {_f(row.get('toplu_usd_1000_paylasim'), 3)} $/1.000 paylaşım" + ("" if r.get("tamam") else " (EKSİK)"))
    if rows:
        t = pd.DataFrame(rows)
        save_table(t, "tablo6b_bdm_gecikme_maliyet_olcumu")
        print(t.round(3).to_string(index=False))
    return rows

In [ ]:
# =============================================================================
# 3. EKSİK BDM'LERİ DEĞERLENDİR, TABLOLARA EKLE, ZİPLE
# =============================================================================
def load_hit_vector(set_name, model, metin=None):
    """tahminler/mesaj_<set>_<model>_<metin>.csv (ya da BDM) dosyasından mesaj sırasına göre hit1 vektörü."""
    name = f"mesaj_{set_name}_BDM_{model}.csv" if metin is None else f"mesaj_{set_name}_{model}_{metin}.csv"
    p = os.path.join(DIRS["preds"], name)
    if not os.path.exists(p):
        return None
    d = pd.read_csv(p).set_index("id")["hit1"]
    ids = EVAL_SETS[set_name]["id"].tolist()
    return d.reindex(ids).fillna(0).values.astype(float)

def ref_from_table8(set_name):
    """Tablo 8'deki referans yapılandırmayı ('USE / Düz') çözer -> (model, metin, hit1_ref)."""
    p = os.path.join(DIRS["results"], f"tablo8_anlamlilik_en_iyiye_karsi_{set_name}.csv")
    if not os.path.exists(p):
        return None
    t8 = pd.read_csv(p)
    if t8.empty:
        return None
    ref = str(t8["Referans"].iloc[0]); model, tt_tr = [x.strip() for x in ref.rsplit(" / ", 1)]
    return model, TR_TO_TEXT_TYPE.get(tt_tr, tt_tr), float(t8["hit1_ref"].iloc[0]), t8

def upsert_rows(df, new_rows, key_cols):
    """Aynı anahtarlı satırları yerinde günceller, yenileri sona ekler (sıra korunur)."""
    new = pd.DataFrame(new_rows)
    if df is None or df.empty:
        return new
    df = df.copy().reset_index(drop=True)
    for _, r in new.iterrows():
        m = pd.Series(True, index=df.index)
        for k in key_cols:
            m &= (df[k].astype(str) == str(r[k]))
        if m.any():
            i = m[m].index[0]
            for c in r.index:
                if c not in df.columns:
                    df[c] = np.nan
                df.at[i, c] = r[c]
            df = df.drop(index=m[m].index[1:]).reset_index(drop=True)
        else:
            df = pd.concat([df, r.to_frame().T], ignore_index=True)
    return df

COMPLETED, INCOMPLETE = [], []
# --- Protokol tutarlılığı: farklı ayarla yanıtlanmış mesajları önbellekten çıkar (bir kez) ---------------------
PROVIDER_OF = dict((m, p) for p, m in MODELS_TO_COMPLETE + BENCH_MODELS)
REDO_PENDING = set()
for (set_name, model), ids in REDO_IDS.items():
    provider = PROVIDER_OF.get(model)
    if provider is None or set_name not in EVAL_SETS:
        continue
    cp = llm_cache_path(provider, model, set_name); marker = cp + ".redo_done"
    if os.path.exists(marker):
        log(f"{set_name} / {model}: yeniden istem daha önce yapılmış, atlandı"); continue
    if os.path.exists(cp):
        lines = open(cp, encoding="utf-8").read().splitlines()
        drop = set(int(i) for i in ids); kept = []; n_drop = 0
        for ln in lines:
            try:
                r = json.loads(ln)
            except Exception:
                continue
            if int(r.get("id", -1)) in drop:
                n_drop += 1; continue
            kept.append(ln)
        shutil.copy(cp, cp + ".bak")
        with open(cp, "w", encoding="utf-8") as f:
            f.write("\n".join(kept) + ("\n" if kept else ""))
        log(f"{set_name} / {model}: {n_drop} önbellek satırı ({len(drop)} mesaj) farklı protokolle yanıtlandığı için çıkarıldı; yeniden istenecek (yedek: {os.path.basename(cp)}.bak)")
    REDO_PENDING.add((set_name, model))
for set_name, dfe in EVAL_SETS.items():
    p5 = os.path.join(DIRS["results"], f"tablo5_bdm_{set_name}_sayisal.csv")
    t5_num = pd.read_csv(p5) if os.path.exists(p5) else pd.DataFrame()
    done_models = set()
    if not t5_num.empty:
        cov = 1 - t5_num["tahminsiz"].fillna(0) / t5_num["n_mesaj"]
        done_models = set(t5_num.loc[cov >= LLM_MIN_COVERAGE, "model"])
    new_rows = []
    for provider, model in MODELS_TO_COMPLETE:
        if provider == "openai" and not OPENAI_API_KEY:
            log(f"{model}: OPENAI_API_KEY yok, atlandı"); continue
        if provider == "google" and not GOOGLE_API_KEY:
            log(f"{model}: GOOGLE_API_KEY yok, atlandı"); continue
        if provider == "anthropic" and not ANTHROPIC_API_KEY:
            log(f"{model}: ANTHROPIC_API_KEY yok, atlandı"); continue
        if model in done_models and not FORCE_REEVALUATE and (set_name, model) not in REDO_PENDING:
            log(f"{set_name} / {model}: Tablo 5'te tam sonuç var, atlandı (FORCE_REEVALUATE=True ile yenilenir)"); continue
        tag = f"{set_name}_BDM_{model}"
        try:
            cost_before = usage_cost_usd(model) if provider == "anthropic" else other_usage_cost(model)
            top1, ms_per_msg, n_fail, n_missing, stop_reason = llm_evaluate(provider, model, dfe, set_name)
            coverage = 1 - n_missing / max(1, len(dfe))
            if coverage < LLM_MIN_COVERAGE:
                INCOMPLETE.append({"veri": set_name, "saglayici": provider, "model": model, "mesaj": len(dfe), "tahminsiz": n_missing,
                                   "kapsam": round(coverage, 4), "basarisiz_cagri": n_fail, "neden": stop_reason or "başarısız çağrılar"})
                log(f"EKSİK ({tag}): {len(dfe) - n_missing}/{len(dfe)} mesaj yanıtlandı (kapsam {coverage:.1%}); Tablo 5'e ALINMADI. "
                    f"Neden: {stop_reason or 'başarısız çağrılar'}. Sorun giderilince bu defteri yeniden çalıştırın; yanıtlananlar önbellekte.")
                continue
            res, hit1 = evaluate_predictions(dfe, top1, [[p] if p else [] for p in top1], tag)
            cost_now = (usage_cost_usd(model) if provider == "anthropic" else other_usage_cost(model))
            cost = cost_now - (cost_before if cost_before == cost_before else 0.0) if cost_now == cost_now else float("nan")
            if (set_name, model) in REDO_PENDING and not t5_num.empty and (t5_num.model == model).any():
                old_cost = float(t5_num.loc[t5_num.model == model, "maliyet_usd"].iloc[0])
                if old_cost == old_cost:     # önceki oturumun maliyeti + yeniden istenen mesajların maliyeti
                    cost = old_cost + (cost if cost == cost else 0.0)
            res.update({"veri": set_name, "saglayici": provider, "model": model, "ms_mesaj": ms_per_msg, "basarisiz_cagri": n_fail,
                        "tahminsiz": n_missing, "maliyet_usd": cost})
            new_rows.append(res); COMPLETED.append((set_name, model))
            if (set_name, model) in REDO_PENDING:
                open(llm_cache_path(provider, model, set_name) + ".redo_done", "w").write(datetime.datetime.now().isoformat())
            log(f"{tag}: ilçe doğruluğu (makale) {res['ilce_dogruluk_makale']:.3f} | hit@1 {res['hit1']:.3f} | {ms_per_msg:.0f} ms/mesaj | tahminsiz {n_missing} | "
                + (usage_summary(model) if provider == "anthropic" else other_usage_summary(model)))
        except Exception as e:
            log(f"HATA: {tag} değerlendirilemedi: {e}")
    if not new_rows:
        continue
    # --- Tablo 5 (sayısal + biçimli) --------------------------------------------------------
    # eksik/yarım satırları (kapsamı düşük) tablodan çıkar, yenileri ekle
    if not t5_num.empty:
        cov = 1 - t5_num["tahminsiz"].fillna(0) / t5_num["n_mesaj"]
        t5_num = t5_num[cov >= LLM_MIN_COVERAGE]
    t5_num = upsert_rows(t5_num, new_rows, ["model"])
    t5_num.to_csv(p5, index=False, encoding="utf-8-sig")
    t5 = pd.DataFrame({"Model": t5_num["model"], "Sağlayıcı": t5_num["saglayici"],
                       "İlçe doğruluğu (makale ölçütü)": [f"{fmt_tr(v)} [{fmt_tr(l)}; {fmt_tr(h)}]" for v, l, h in zip(t5_num.ilce_dogruluk_makale, t5_num.ilce_ga_alt, t5_num.ilce_ga_ust)],
                       "Mesaj hit@1": [f"{fmt_tr(v)} [{fmt_tr(l)}; {fmt_tr(h)}]" for v, l, h in zip(t5_num.hit1, t5_num.hit1_ga_alt, t5_num.hit1_ga_ust)],
                       "hit@1 (ilçe ort.)": [fmt_tr(v) for v in t5_num.hit1_ilce_ort], "ms/mesaj": [fmt_tr(v, 0) for v in t5_num.ms_mesaj],
                       "Tahminsiz": t5_num.tahminsiz.values, "Maliyet (USD)": [fmt_tr(v, 2) for v in t5_num.maliyet_usd]})
    save_table(t5, f"tablo5_bdm_{set_name}")
    print(f"\n[{set_name}] Tablo 5 (güncel):\n", t5.to_string(index=False))
    # --- Tablo 8: en iyi yerel yapılandırmaya karşı eşleştirilmiş testler (Holm tüm satırlar için yeniden) ----
    r8 = ref_from_table8(set_name)
    if r8:
        ref_model, ref_tt, hit1_ref, t8 = r8
        a = load_hit_vector(set_name, ref_model, ref_tt)
        if a is not None:
            rows8 = []
            for r in new_rows:
                b = load_hit_vector(set_name, r["model"])
                if b is None or len(b) != len(a):
                    continue
                p, n10, n01 = mcnemar_exact(a, b); d, lo, hi = paired_bootstrap(a, b)
                rows8.append({"Referans": f"{ref_model} / {TEXT_TYPE_TR[ref_tt]}", "Karşılaştırılan": f"{r['model']} (BDM)", "hit1_ref": hit1_ref, "hit1_diger": float(r["hit1"]),
                              "fark": d, "fark_GA_alt": lo, "fark_GA_ust": hi, "sadece_ref_dogru": n10, "sadece_diger_dogru": n01, "p_McNemar": p})
            if rows8:
                keep = t8[~t8["Karşılaştırılan"].isin([x["Karşılaştırılan"] for x in rows8])]
                t8n = pd.concat([keep.drop(columns=[c for c in ("p_Holm", "anlamli_0.05") if c in keep.columns]), pd.DataFrame(rows8)], ignore_index=True)
                # Tablo 5'te (tam) bulunmayan BDM satırlarını at (eksik ya da eski model adları)
                ok_bdm = {f"{m} (BDM)" for m in t5_num["model"]}
                is_bdm = t8n["Karşılaştırılan"].str.endswith("(BDM)")
                t8n = t8n[~is_bdm | t8n["Karşılaştırılan"].isin(ok_bdm)].reset_index(drop=True)
                t8n["p_Holm"] = holm(t8n["p_McNemar"]); t8n["anlamli_0.05"] = t8n["p_Holm"] < 0.05
                save_table(t8n, f"tablo8_anlamlilik_en_iyiye_karsi_{set_name}")
                log(f"Tablo 8 ({set_name}) güncellendi: {len(rows8)} yeni satır, Holm yeniden hesaplandı.")
    # --- Tablo 7: açık / karma / örtük (yalnız sentetik) ---------------------------------------------
    if set_name == "sentetik" and "explicitness" in dfe.columns:
        p7 = os.path.join(DIRS["results"], "tablo7_acik_ortuk_hit1.csv")
        t7 = pd.read_csv(p7) if os.path.exists(p7) else pd.DataFrame(columns=["Model", "Açık", "Karma", "Örtük"])
        rows7 = []
        expl = dfe.set_index("id")["explicitness"]
        for r in new_rows:
            b = pd.read_csv(os.path.join(DIRS["preds"], f"mesaj_{set_name}_BDM_{r['model']}.csv")).set_index("id")["hit1"].reindex(expl.index)
            g = b.groupby(expl.values).mean()
            rows7.append({"Model": f"{r['model']} (BDM)", "Açık": g.get("explicit", np.nan), "Karma": g.get("mixed", np.nan), "Örtük": g.get("implicit", np.nan)})
        ok_bdm = {f"{m} (BDM)" for m in t5_num["model"]}
        t7 = t7[~t7["Model"].str.endswith("(BDM)") | t7["Model"].isin(ok_bdm)]
        t7 = upsert_rows(t7, rows7, ["Model"])
        save_table(t7, "tablo7_acik_ortuk_hit1")
        log("Tablo 7 güncellendi.")

# --- Gecikme ve maliyet ölçümü (Tablo 6b) ---------------------------------------------------------------
if RUN_LATENCY_BENCHMARK and "sentetik" in EVAL_SETS:
    log(f"Gecikme/maliyet ölçümü: her BDM için {BENCH_BATCH_REQUESTS} toplu + {BENCH_SINGLE_REQUESTS} tekil istek (aynı paylaşımlar, sıralı)")
    run_latency_benchmarks(BENCH_MODELS, EVAL_SETS["sentetik"])

# --- Şekil 2(b) için örnek: tek paylaşım, 66 ihtiyaç, SBERT-mpnet kosinüs benzerliği ----------------------
def _plain_fig(s_):
    s_ = re.sub(r"[^a-zA-ZğĞçÇşŞüÜöÖıİ]", " ", str(s_))
    return " ".join(s_.replace("İ", "i").lower().split())
if MAKE_FIG2_EXAMPLE:
    try:
        from sentence_transformers import SentenceTransformer
        import torch
        r_ = df_syn.set_index("id").loc[FIG2_EXAMPLE_ID]
        if re.search(r"\d", str(r_["English"])):
            log("Şekil 2 örneğinde rakam var; düz metin ön işlemesi birebir uygulanamayabilir")
        m_ = SentenceTransformer(FIG2_MODEL, device="cuda" if torch.cuda.is_available() else "cpu")
        E_ = m_.encode([_plain_fig(r_["English"])] + [_plain_fig(n) for n in EN_LABELS], normalize_embeddings=True)
        sc_ = E_[1:] @ E_[0]
        order_ = np.argsort(-sc_)[:5]
        top_ = [{"en": EN_LABELS[i], "tr": EN2TR.get(EN_LABELS[i], EN_LABELS[i]), "skor": round(float(sc_[i]), 4)} for i in order_]
        pipe_top1 = None
        pf = os.path.join(DIRS["preds"], "mesaj_sentetik_SBERT-mpnet_plain.csv")
        if os.path.exists(pf):
            pipe_top1 = pd.read_csv(pf).set_index("id").loc[FIG2_EXAMPLE_ID, "top1"]
        ex_ = {"id": int(FIG2_EXAMPLE_ID), "ilce": r_["district"], "metin_en": r_["English"], "metin_tr": r_["Message"],
               "gercek": [x for x in (r_["label1"], r_["label2"], r_["label3"]) if isinstance(x, str)],
               "model": FIG2_MODEL, "model_label": "SBERT-mpnet", "on_isleme": "düz metin", "top": top_,
               "boru_hatti_top1": pipe_top1, "boru_hatti_ile_ayni": (pipe_top1 == top_[0]["en"]) if pipe_top1 is not None else None}
        with open(os.path.join(DIRS["results"], "sekil2_ornek.json"), "w", encoding="utf-8") as f_:
            json.dump(ex_, f_, ensure_ascii=False, indent=1)
        log(f"Şekil 2 örneği: {[(x['tr'], x['skor']) for x in top_]} | boru hattı top-1: {pipe_top1} | aynı: {ex_['boru_hatti_ile_ayni']}")
    except Exception as e:
        log("Şekil 2 örneği üretilemedi (makalede yalnız (a) paneli kullanılır):", repr(e)[:200])

# eksikler
pe = os.path.join(DIRS["results"], "tablo5_bdm_eksik.csv")
if INCOMPLETE:
    pd.DataFrame(INCOMPLETE).to_csv(pe, index=False, encoding="utf-8-sig")
    log(f"Eksik kalan BDM değerlendirmeleri: {[(x['veri'], x['model'], x['kapsam']) for x in INCOMPLETE]} -> {pe}")
elif os.path.exists(pe):
    os.remove(pe)

# tahmin dosyalarından eksik modellere ait (yarım) olanları temizle: yalnız Tablo 5'te olmayan BDM'ler
for set_name in EVAL_SETS:
    p5 = os.path.join(DIRS["results"], f"tablo5_bdm_{set_name}_sayisal.csv")
    if not os.path.exists(p5):
        continue
    ok_models = set(pd.read_csv(p5)["model"])
    for f in glob.glob(os.path.join(DIRS["preds"], f"*_{set_name}_BDM_*.csv")):
        m = re.search(rf"_{set_name}_BDM_(.+)\.csv$", os.path.basename(f)).group(1)
        if m not in ok_models:
            os.rename(f, f + ".eksik")   # silinmez, yeniden adlandırılır

with open(os.path.join(DIRS["logs"], f"bdm_ek_{datetime.datetime.now().strftime('%Y%m%d_%H%M')}.json"), "w", encoding="utf-8") as f:
    json.dump({"tamamlanan": COMPLETED, "eksik": INCOMPLETE, "modeller": MODELS_TO_COMPLETE, "yeniden_istenen": [list(k) for k in REDO_PENDING],
               "anthropic_kullanim": {m: dict(u) for m, u in USAGE.items()}, "diger_kullanim": {m: dict(u) for m, u in OTHER_USAGE.items()}}, f, ensure_ascii=False, indent=1)

# --- Zip ---------------------------------------------------------------------------------
zip_base = os.path.join(BASE_DIR, f"sonuclar_{RUN_TAG}_ek")
staging = os.path.join(LOCAL_SCRATCH, f"zip_{RUN_TAG}_ek")
shutil.rmtree(staging, ignore_errors=True)
shutil.copytree(RUN_DIR, os.path.join(staging, RUN_TAG))
os.makedirs(os.path.join(staging, RUN_TAG, "data"), exist_ok=True)
for f in glob.glob(os.path.join(DATA_DIR, f"sentetik_paylasimlar_{GEN_SIGNATURE}.*")):
    shutil.copy(f, os.path.join(staging, RUN_TAG, "data", os.path.basename(f)))
geo = os.path.join(DATA_DIR, "geo", "istanbul_ilceler.geojson")
if os.path.exists(geo):
    os.makedirs(os.path.join(staging, RUN_TAG, "data", "geo"), exist_ok=True); shutil.copy(geo, os.path.join(staging, RUN_TAG, "data", "geo", "istanbul_ilceler.geojson"))
if os.path.exists(zip_base + ".zip"):
    os.remove(zip_base + ".zip")
zip_path = shutil.make_archive(zip_base, "zip", staging)
log(f"Zip hazır: {zip_path} ({os.path.getsize(zip_path) / 1e6:.1f} MB) | tamamlanan: {COMPLETED} | eksik: {[(x['veri'], x['model']) for x in INCOMPLETE]}")
if IN_COLAB and AUTO_DOWNLOAD_ZIP:
    try:
        from google.colab import files
        files.download(zip_path)
    except Exception as e:
        log("Otomatik indirme başarısız (Drive'daki zip kullanılabilir):", e)